# Keyword baseline (Issue #12, Task B)

A non-ML baseline: for each of the 40 categories, write down a short list of words/phrases
you'd expect to see in that clause, then predict "this category is present" whenever one of
those words shows up in a chunk. No training, no model weights — just string matching.

This exists so the real classifier (TF-IDF baseline, transformer fine-tune) has something to
beat. If a simple keyword rule already gets good precision/recall on a category, that's a
cheap signal; if the trained model can't beat this, that's a red flag worth investigating.

## Step 1 — Load, clean, and chunk (self-contained)

Loads directly from the raw `data/cuad/` JSON and does the load -> clean -> chunk steps
inline, rather than depending on `data/processed/` files generated by other notebooks. This
way anyone can open just this notebook and run it top to bottom with no setup besides the
raw data already in the repo.

- **Load**: same SQuAD-style flatten as `main.ipynb` (one row per annotated span).
- **Clean**: drop the one duplicate contract (`main.ipynb`, issue #7).
- **Chunk**: same 510-token window / 256-token overlap as `11_chunking_strategy.ipynb`
  (issue #11) - a chunk is labeled with a category only if the full annotated span fits
  inside it.

In [1]:
import json as _json
import re
from pathlib import Path

import pandas as pd
from transformers import AutoTokenizer

DATA_DIR = Path("../data/cuad")
TRAIN_PATH = DATA_DIR / "train_separate_questions.json"
TEST_PATH = DATA_DIR / "test.json"


def category_from_id(qid):
    return re.sub(r"_\d+$", "", qid.split("__")[-1])


def load_cuad(path):
    with open(path) as f:
        data = _json.load(f)["data"]
    clause_rows, contract_rows = [], []
    for c in data:
        cid = c["title"]
        for p in c["paragraphs"]:
            contract_rows.append({"contract_id": cid, "text": p["context"]})
            for qa in p["qas"]:
                cat = category_from_id(qa["id"])
                for ans in qa["answers"]:
                    start = ans["answer_start"]
                    clause_rows.append({
                        "contract_id": cid, "category": cat, "clause_text": ans["text"],
                        "start": start, "end": start + len(ans["text"]),
                    })
    return pd.DataFrame(clause_rows), pd.DataFrame(contract_rows)


def drop_duplicate_contracts(clauses, contracts):
    keep = contracts.drop_duplicates("text", keep="first")
    dropped = set(contracts["contract_id"]) - set(keep["contract_id"])
    return clauses[~clauses["contract_id"].isin(dropped)].copy(), keep.copy()


# --- load + clean (matches main.ipynb) --------------------------------------
train_clauses, train_contracts = load_cuad(TRAIN_PATH)
test_clauses, test_contracts = load_cuad(TEST_PATH)
train_clauses, train_contracts = drop_duplicate_contracts(train_clauses, train_contracts)
test_clauses, test_contracts = drop_duplicate_contracts(test_clauses, test_contracts)

print(f"train: {len(train_contracts)} contracts / {len(train_clauses)} spans")
print(f"test:  {len(test_contracts)} contracts / {len(test_clauses)} spans")

# --- chunk (matches 11_chunking_strategy.ipynb: window=510, overlap=256) ----
tokenizer = AutoTokenizer.from_pretrained("distilroberta-base")


def chunk_contract(text, window_size=510, overlap=256):
    encoding = tokenizer(text, add_special_tokens=False, return_offsets_mapping=True)
    token_ids = encoding["input_ids"]
    offsets = encoding["offset_mapping"]
    step = window_size - overlap
    chunks = []
    for token_start in range(0, len(token_ids), step):
        token_end = min(token_start + window_size, len(token_ids))
        chunk_offsets = offsets[token_start:token_end]
        if not chunk_offsets:
            break
        char_start, char_end = chunk_offsets[0][0], chunk_offsets[-1][1]
        chunks.append({"char_start": char_start, "char_end": char_end, "text": text[char_start:char_end]})
        if token_end == len(token_ids):
            break
    return chunks


def build_labeled_chunks(contracts, clauses):
    rows = []
    clauses_by_contract = {cid: g for cid, g in clauses.groupby("contract_id")}
    for _, contract in contracts.iterrows():
        cid = contract["contract_id"]
        chunks = chunk_contract(contract["text"])
        contract_clauses = clauses_by_contract.get(cid)
        for chunk in chunks:
            labels = []
            if contract_clauses is not None:
                for _, clause in contract_clauses.iterrows():
                    if chunk["char_start"] <= clause["start"] and clause["end"] <= chunk["char_end"]:
                        labels.append(clause["category"])
            rows.append({"contract_id": cid, "text": chunk["text"], "labels": sorted(set(labels))})
    return pd.DataFrame(rows)


print("\nChunking train...")
train_chunks = build_labeled_chunks(train_contracts, train_clauses)
print("Chunking test...")
test_chunks = build_labeled_chunks(test_contracts, test_clauses)
print(f"train chunks: {len(train_chunks)} | test chunks: {len(test_chunks)}")

train: 407 contracts / 11166 spans
test:  102 contracts / 2643 spans


[transformers] Token indices sequence length is longer than the specified maximum sequence length for this model (23248 > 512). Running this sequence through the model will result in indexing errors



Chunking train...
Chunking test...
train chunks: 20997 | test chunks: 4283


## Step 2 — Build a keyword list per category

For each category: start from its description in `category_descriptions.csv`, then skim a
handful of actual positive clause examples (`train_clauses`, built in Step 1, filtered by
category) to pull out words/phrases that actually show up in real contract language, not
just the formal definition.

Example shape: `{"Source Code Escrow": ["escrow", "escrow agent", "source code"], ...}`

In [2]:
import csv
import re
from collections import Counter

# category_descriptions.csv uses slightly different casing than the category names
# derived from the JSON question ids (e.g. "Cap on Liability" vs "Cap On Liability") -
# reconcile via a normalized key, same approach as 14_label_selection.ipynb
def normalize(name):
    return re.sub(r"[^a-z0-9]", "", name.lower())

with open(DATA_DIR / "category_descriptions.csv", encoding="utf-8-sig") as f:
    rows = list(csv.DictReader(f))

official_categories = [r["Category (incl. context and answer)"].replace("Category: ", "") for r in rows]
descriptions = {
    r["Category (incl. context and answer)"].replace("Category: ", ""):
        r["Description"].replace("Description: ", "").strip()
    for r in rows
}
norm_to_official = {normalize(c): c for c in official_categories}
train_clauses["category"] = train_clauses["category"].map(
    lambda c: norm_to_official.get(normalize(c), c)
)

# generic contract/legal boilerplate that shows up everywhere and isn't a useful signal
STOPWORDS = set("""
a an the this that these those and or but if then else when while of to in on at by for with
without from as is are was were be been being have has had do does did shall will would may
might must can could should not no nor so such it its they them their he she his her you your
we our i me my itself himself herself themselves which who whom whose what where why how all
any both each few more most other some such than too very s t just don now also upon under
between into through during before after above below out up down over again further once
here there party parties agreement contract section clause pursuant hereby hereof herein
hereunder thereof thereto whereas provided provide provides shall notwithstanding
""".split())

WORD_RE = re.compile(r"[a-z]{3,}")


def extract_words(text):
    return [w for w in WORD_RE.findall(text.lower()) if w not in STOPWORDS]


# background rate: how often each word shows up across ALL positive clauses (any category) -
# used to downweight generic words and surface ones that are distinctive to one category
background_counts = Counter()
for text in train_clauses["clause_text"]:
    background_counts.update(set(extract_words(text)))
n_clauses_total = len(train_clauses)

MIN_OCCURRENCES = 3  # a candidate word must show up in at least this many positive clauses
TOP_N = 8

draft_keywords = {}
for category in official_categories:
    texts = train_clauses.loc[train_clauses["category"] == category, "clause_text"]
    category_counts = Counter()
    for text in texts:
        category_counts.update(set(extract_words(text)))
    n_cat = len(texts)

    scored = []
    for word, count in category_counts.items():
        if count < MIN_OCCURRENCES:
            continue
        cat_freq = count / n_cat if n_cat else 0
        bg_freq = background_counts[word] / n_clauses_total
        lift = cat_freq / bg_freq if bg_freq > 0 else 0
        scored.append((word, count, round(lift, 1)))

    scored.sort(key=lambda x: -x[2])
    draft_keywords[category] = scored[:TOP_N]

# print everything for manual review - nothing here is final
for category in official_categories:
    print(f"=== {category} ({descriptions[category][:70]}) ===")
    for word, count, lift in draft_keywords[category]:
        print(f"  {word:20s} in {count:3d} clauses  lift={lift}")
    if not draft_keywords[category]:
        print("  (no candidates cleared the occurrence threshold)")
    print()

=== Document Name (The name of the contract) ===
  customization        in   4 clauses  lift=21.4
  distributorship      in   3 clauses  lift=20.0
  branding             in  16 clauses  lift=17.1
  strategic            in  25 clauses  lift=15.5
  hosting              in  16 clauses  lift=15.3
  outsourcing          in  16 clauses  lift=14.2
  alliance             in  24 clauses  lift=13.9
  sponsorship          in  25 clauses  lift=11.3

=== Parties (The two or more parties who signed the contract) ===
  nissin               in   6 clauses  lift=5.6
  veoneer              in   3 clauses  lift=5.6
  deutsche             in   4 clauses  lift=5.6
  pty                  in   4 clauses  lift=5.6
  plc                  in   5 clauses  lift=5.6
  yisheng              in   3 clauses  lift=5.6
  sometimes            in  15 clauses  lift=5.6
  gmbh                 in   5 clauses  lift=5.6

=== Agreement Date (The date of the contract) ===
  november             in  28 clauses  lift=13.9
  april 

## Step 3 — Apply the rule

For each chunk, for each category: predict "present" if any of that category's keywords
appear in the chunk text (case-insensitive substring or word-boundary match), else "absent".
Run this on the **test** chunks only — the keyword lists themselves came from train, so
scoring on test keeps this comparable to the other baselines.

In [3]:
import re

# test_chunks["labels"] came from main.ipynb's raw JSON-derived category names (e.g.
# "Cap On Liability"), doesn't always match category_descriptions.csv's casing
# ("Cap on Liability"). without this, every mismatched category silently scores 0 support.
test_chunks["labels"] = test_chunks["labels"].apply(
    lambda labels: [norm_to_official.get(normalize(c), c) for c in labels]
)

# Price Restrictions has zero positive examples in the official test set (confirmed in
# main.ipynb) - excluded from evaluation everywhere else on this branch, so we match that
# here too. It still has a draft keyword list above; just not scored below.
EVAL_CATEGORIES = [c for c in official_categories if c != "Price Restrictions"]


def matches_any_keyword(text, keywords):
    text_lower = text.lower()
    return any(
        re.search(r"\b" + re.escape(word) + r"\b", text_lower)
        for word, _count, _lift in keywords
    )


predictions = pd.DataFrame(index=test_chunks.index, columns=EVAL_CATEGORIES, dtype=int)
for category in EVAL_CATEGORIES:
    keywords = draft_keywords.get(category, [])
    predictions[category] = test_chunks["text"].apply(
        lambda t: int(matches_any_keyword(t, keywords))
    )

ground_truth = pd.DataFrame(index=test_chunks.index, columns=EVAL_CATEGORIES, dtype=int)
for category in EVAL_CATEGORIES:
    ground_truth[category] = test_chunks["labels"].apply(
        lambda labels: int(category in labels)
    )

print(f"Test chunks: {len(test_chunks)}")
print(f"Chunks predicted positive for >=1 category: {(predictions.sum(axis=1) > 0).sum()}")
print(f"Chunks with >=1 true category (ground truth): {(ground_truth.sum(axis=1) > 0).sum()}")

Test chunks: 4283
Chunks predicted positive for >=1 category: 3615
Chunks with >=1 true category (ground truth): 1693


## Step 4 — Evaluate

Per-category precision / recall / F1, plus macro and micro averages — same metrics issue #12
asks for, so this is directly comparable to the TF-IDF baseline (Task A).

In [4]:
def precision_recall_f1(y_true, y_pred):
    tp = int(((y_true == 1) & (y_pred == 1)).sum())
    fp = int(((y_true == 0) & (y_pred == 1)).sum())
    fn = int(((y_true == 1) & (y_pred == 0)).sum())
    precision = tp / (tp + fp) if (tp + fp) > 0 else 0.0
    recall = tp / (tp + fn) if (tp + fn) > 0 else 0.0
    f1 = 2 * precision * recall / (precision + recall) if (precision + recall) > 0 else 0.0
    return precision, recall, f1, tp, fp, fn


rows = []
tp_total = fp_total = fn_total = 0
for category in EVAL_CATEGORIES:
    p, r, f1, tp, fp, fn = precision_recall_f1(ground_truth[category], predictions[category])
    rows.append({
        "category": category, "precision": round(p, 3), "recall": round(r, 3),
        "f1": round(f1, 3), "tp": tp, "fp": fp, "fn": fn,
        "support": int(ground_truth[category].sum()),
    })
    tp_total += tp
    fp_total += fp
    fn_total += fn

results_df = pd.DataFrame(rows).sort_values("f1", ascending=False)

macro_precision = results_df["precision"].mean()
macro_recall = results_df["recall"].mean()
macro_f1 = results_df["f1"].mean()

micro_precision = tp_total / (tp_total + fp_total) if (tp_total + fp_total) > 0 else 0.0
micro_recall = tp_total / (tp_total + fn_total) if (tp_total + fn_total) > 0 else 0.0
micro_f1 = (
    2 * micro_precision * micro_recall / (micro_precision + micro_recall)
    if (micro_precision + micro_recall) > 0 else 0.0
)

# everything printed here, in this notebook - nothing written to disk for teammates to
# dig up separately; this cell's output IS the comparison point for the other models.
print("Per-category results (sorted by F1):")
display(results_df)

print(f"\nMacro  -  P: {macro_precision:.3f}  R: {macro_recall:.3f}  F1: {macro_f1:.3f}")
print(f"Micro  -  P: {micro_precision:.3f}  R: {micro_recall:.3f}  F1: {micro_f1:.3f}")

Per-category results (sorted by F1):


,category,precision,recall,f1,tp,fp,fn,support
25,Non-Transferable License,0.571,0.632,0.600,36,27,21,57
4,Expiration Date,0.388,0.493,0.435,73,115,75,148
7,Governing Law,0.297,0.586,0.394,92,218,65,157
29,Irrevocable or Perpetual License,0.214,0.931,0.348,27,99,2,29
32,Audit Rights,0.271,0.302,0.286,35,94,81,116
9,Non-Compete,0.224,0.381,0.282,24,83,39,63
38,Covenant Not to Sue,0.250,0.288,0.268,19,57,47,66
2,Agreement Date,0.162,0.491,0.244,55,284,57,112
21,Volume Restriction,0.165,0.342,0.222,13,66,25,38
37,Insurance,0.176,0.278,0.216,25,117,65,90



Macro  -  P: 0.132  R: 0.424  F1: 0.157
Micro  -  P: 0.098  R: 0.303  F1: 0.148
